## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, pickle, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import log_loss, average_precision_score

## Setup

In [2]:
SEED = 42
np.random.seed(SEED)
drop_rng = np.random.RandomState(999)  # separate dropout stream
K, REPEATS = 5, 1                      # scratch twin: one repeat only (repeat 1 = the same folds as the pytorch twin)
FOLDS = [int(f) for f in os.environ.get("KF_FOLDS", "1,2,3,4,5").split(",")]   # lets several jobs each take some folds
KP = "weights_V4_1/classification_pytorch_kfold_metrics.json"
assert os.path.exists(KP), "pytorch K-Fold twin must run first (epochs and threshold come from it)"
ORA = {(r["model"], r["repeat"], r["fold"]): r for r in json.load(open(KP))["folds"]}
os.makedirs("weights_V4_1", exist_ok=True)
VARS = {"ReLU (baseline)": dict(ffn="relu", hid=64), "ReGLU, param-matched": dict(ffn="reglu", hid=43)}
NAME = "FT-Transformer-lite | "

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
IN_DIM, D, H, DK, LAYERS, P = len(FEATURE_COLS), 32, 4, 8, 2, 0.1
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {IN_DIM} dims | d={D} L={LAYERS} H={H} p={P}")

features: 17 numeric + 5 dummies = 22 dims | d=32 L=2 H=4 p=0.1


## Dataloader

In [4]:
X_all = X_full.values
y_all = y_full.astype(int)
inc_idx = FEATURE_COLS.index("Income")

def prep_split(tr_idx, te_idx):
    """Median-fill Income and standardise using the training rows only."""
    Xtr, Xte = X_all[tr_idx].copy(), X_all[te_idx].copy()
    med = np.nanmedian(Xtr[:, inc_idx])
    for a in (Xtr, Xte):
        a[:, inc_idx] = np.where(np.isnan(a[:, inc_idx]), med, a[:, inc_idx])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte), y_all[tr_idx], y_all[te_idx]

D, H, DK, LAYERS, P = 32, 4, 8, 2, 0.1
IN_DIM = len(FEATURE_COLS)
PW = 1.0

## Model

In [5]:
EPS = 1e-5
def ln_fwd(x, g, b):
    mu = x.mean(-1, keepdims=True); xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    xh = xc / np.sqrt(var + EPS); y = xh * g + b
    return y, (x, xh, xc, var, g)
def ln_bwd(dy, c):
    x, xh, xc, var, g = c; N = x.shape[-1]; iv = 1.0 / np.sqrt(var + EPS)
    dxh = dy * g
    dvar = (dxh * xc * -0.5 * (var + EPS) ** (-1.5)).sum(-1, keepdims=True)
    dmu = (-dxh * iv).sum(-1, keepdims=True) + dvar * (-2.0 * xc).mean(-1, keepdims=True)
    dx = dxh * iv + dvar * 2.0 * xc / N + dmu / N
    return dx, (dy * xh).sum(tuple(range(dy.ndim - 1))), dy.sum(tuple(range(dy.ndim - 1)))
def lin_fwd(x, W, b): return x @ W + b
def dropout_fwd(x, train):
    if not train: return x, None
    m = (drop_rng.rand(*x.shape) > P) / (1.0 - P); return x * m, m
def softmax_fwd(s):
    e = np.exp(s - s.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

In [6]:
def init_params(ffn, hid):
    """Scratch parameters. ReLU: F1 is (d, hid). ReGLU: F1 is (d, 2*hid) = [value | gate]."""
    global FFN, HID, FFW, P_
    FFN, HID = ffn, hid; FFW = hid if ffn == "relu" else 2 * hid
    P_ = {}
    P_["tokW"] = np.random.randn(IN_DIM, D) * np.sqrt(2.0 / 1.0)
    P_["tokb"] = np.zeros((IN_DIM, D))
    P_["cls"] = np.random.randn(D) * 0.02
    for l in range(LAYERS):
        P_[f"Wq{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bq{l}"] = np.zeros(D)
        P_[f"Wk{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bk{l}"] = np.zeros(D)
        P_[f"Wv{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bv{l}"] = np.zeros(D)
        P_[f"Wo{l}"] = np.random.randn(D, D) * np.sqrt(2.0 / D); P_[f"bo{l}"] = np.zeros(D)
        P_[f"F1{l}"] = np.random.randn(D, FFW) * np.sqrt(2.0 / D); P_[f"f1b{l}"] = np.zeros(FFW)
        P_[f"F2{l}"] = np.random.randn(HID, D) * np.sqrt(2.0 / HID); P_[f"f2b{l}"] = np.zeros(D)
        P_[f"g1{l}"] = np.ones(D); P_[f"b1{l}"] = np.zeros(D)
        P_[f"g2{l}"] = np.ones(D); P_[f"b2{l}"] = np.zeros(D)
    P_["gf"] = np.ones(D); P_["bf"] = np.zeros(D)
    P_["Wh"] = np.random.randn(D, 1) * np.sqrt(2.0 / D); P_["bh"] = np.zeros(1)
    return P_
for v, s in VARS.items():
    print(f"{v:22s} scratch parameters: {sum(a.size for a in init_params(**s).values())}")

ReLU (baseline)        scratch parameters: 18625
ReGLU, param-matched   scratch parameters: 18733


In [7]:
def block_fwd(t, l, train):
    y1, c1 = ln_fwd(t, P_[f"g1{l}"], P_[f"b1{l}"]); B, T, Dd = y1.shape
    Q = lin_fwd(y1, P_[f"Wq{l}"], P_[f"bq{l}"]); K = lin_fwd(y1, P_[f"Wk{l}"], P_[f"bk{l}"]); V = lin_fwd(y1, P_[f"Wv{l}"], P_[f"bv{l}"])
    Q4 = Q.reshape(B, T, H, DK).transpose(0, 2, 1, 3); K4 = K.reshape(B, T, H, DK).transpose(0, 2, 1, 3); V4 = V.reshape(B, T, H, DK).transpose(0, 2, 1, 3)
    S = (Q4 @ K4.transpose(0, 1, 3, 2)) / np.sqrt(DK); Pb = softmax_fwd(S)
    Pm, dmask = dropout_fwd(Pb, train)
    O4 = Pm @ V4; O = O4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    u = t + lin_fwd(O, P_[f"Wo{l}"], P_[f"bo{l}"])
    y2, c2 = ln_fwd(u, P_[f"g2{l}"], P_[f"b2{l}"])
    h1 = lin_fwd(y2, P_[f"F1{l}"], P_[f"f1b{l}"])
    if FFN == "reglu": a1 = h1[..., :HID] * np.maximum(h1[..., HID:], 0)        # value * ReLU(gate)
    else: a1 = np.maximum(h1, 0)
    ad, amask = dropout_fwd(a1, train)
    f = lin_fwd(ad, P_[f"F2{l}"], P_[f"f2b{l}"]); v = u + f
    return v, {"t": t, "y1": y1, "c1": c1, "Q4": Q4, "K4": K4, "V4": V4, "S": S, "Pb": Pb, "Pm": Pm,
               "dmask": dmask, "O": O, "u": u, "y2": y2, "c2": c2, "h1": h1, "amask": amask, "ad": ad, "l": l}

def block_bwd(dv, c, G):
    l = c["l"]; y1, Q4, K4, V4, Pb, Pm = c["y1"], c["Q4"], c["K4"], c["V4"], c["Pb"], c["Pm"]
    B, T, Dd = y1.shape
    # FFN path: v = u + f
    df = dv.copy(); du_res = dv.copy()
    d_ad = df @ P_[f"F2{l}"].T; G[f"F2{l}"] += c["ad"].reshape(-1, HID).T @ df.reshape(-1, Dd); G[f"f2b{l}"] += df.sum((0, 1))
    if c["amask"] is not None: d_ad = d_ad * c["amask"]
    h1 = c["h1"]
    if FFN == "reglu":
        val, gate = h1[..., :HID], h1[..., HID:]
        d_h1 = np.concatenate([d_ad * np.maximum(gate, 0), d_ad * val * (gate > 0)], axis=-1)
    else:
        d_h1 = d_ad * (h1 > 0)
    d_y2 = d_h1 @ P_[f"F1{l}"].T; G[f"F1{l}"] += c["y2"].reshape(-1, Dd).T @ d_h1.reshape(-1, FFW); G[f"f1b{l}"] += d_h1.sum((0, 1))
    du_ffn, dg2, db2 = ln_bwd(d_y2, c["c2"]); G[f"g2{l}"] += dg2; G[f"b2{l}"] += db2
    du = du_res + du_ffn
    # MHSA path: u = t + Wo(O)
    dt_res = du.copy()
    d_O = du @ P_[f"Wo{l}"].T; G[f"Wo{l}"] += c["O"].reshape(-1, Dd).T @ du.reshape(-1, Dd); G[f"bo{l}"] += du.sum((0, 1))
    dO4 = d_O.reshape(B, T, H, DK).transpose(0, 2, 1, 3)
    dV4 = np.matmul(Pm.transpose(0, 1, 3, 2), dO4)
    dPm = np.matmul(dO4, V4.transpose(0, 1, 3, 2))
    dPm = dPm * c["dmask"] if c["dmask"] is not None else dPm
    dS = Pb * (dPm - (dPm * Pb).sum(-1, keepdims=True))
    dS = dS / np.sqrt(DK)
    dQ4 = np.matmul(dS, K4); dK4 = np.matmul(dS.transpose(0, 1, 3, 2), Q4)
    dQ = dQ4.transpose(0, 2, 1, 3).reshape(B, T, Dd); dK = dK4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    dV = dV4.transpose(0, 2, 1, 3).reshape(B, T, Dd)
    d_y1 = dQ @ P_[f"Wq{l}"].T + dK @ P_[f"Wk{l}"].T + dV @ P_[f"Wv{l}"].T
    G[f"Wq{l}"] += y1.reshape(-1, Dd).T @ dQ.reshape(-1, Dd); G[f"bq{l}"] += dQ.sum((0, 1))
    G[f"Wk{l}"] += y1.reshape(-1, Dd).T @ dK.reshape(-1, Dd); G[f"bk{l}"] += dK.sum((0, 1))
    G[f"Wv{l}"] += y1.reshape(-1, Dd).T @ dV.reshape(-1, Dd); G[f"bv{l}"] += dV.sum((0, 1))
    dt_mhsa, dg1, db1 = ln_bwd(d_y1, c["c1"]); G[f"g1{l}"] += dg1; G[f"b1{l}"] += db1
    return dt_res + dt_mhsa

In [8]:
def model_fwd(Xb, train):
    B = Xb.shape[0]
    toks = [Xb[:, j:j+1] @ P_["tokW"][j:j+1, :] + P_["tokb"][j:j+1, :] for j in range(IN_DIM)]
    t = np.stack(toks, axis=1)
    t = np.concatenate([np.broadcast_to(P_["cls"], (B, 1, D)), t], axis=1)
    caches = []
    for l in range(LAYERS):
        t, c = block_fwd(t, l, train); caches.append(c)
    yf, cf = ln_fwd(t[:, 0], P_["gf"], P_["bf"])
    logit = yf @ P_["Wh"] + P_["bh"]
    return logit, (Xb, caches, yf, cf)

def model_bwd(dlogit, cache, G):
    Xb, caches, yf, cf = cache
    G["Wh"] += yf.T @ dlogit; G["bh"] += dlogit.sum(0)
    d_yf = dlogit @ P_["Wh"].T
    d_cls, dg, db = ln_bwd(d_yf, cf); G["gf"] += dg; G["bf"] += db
    B = Xb.shape[0]; dt = np.zeros((B, 1 + IN_DIM, D))
    dt[:, 0] = d_cls
    for l in reversed(range(LAYERS)):
        dt = block_bwd(dt, caches[l], G)
    G["cls"] += dt[:, 0].sum(0)
    dTok = dt[:, 1:]
    for j in range(IN_DIM):
        G["tokW"][j:j+1, :] += (Xb[:, j:j+1].T @ dTok[:, j, :])
        G["tokb"][j:j+1, :] += dTok[:, j, :].sum(0, keepdims=True)

In [9]:
def fwd_loss(Xb, yb, train, weighted):
    logit, cache = model_fwd(Xb, train)
    p = sigmoid(logit).ravel(); y = yb.ravel(); B = len(y)
    eps = 1e-12; pc = np.clip(p, eps, 1 - eps)
    if weighted:
        loss = -(PW * y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()
        dlogit = ((p + p * y * (PW - 1) - PW * y) / B).reshape(-1, 1)
    else:
        loss = -(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()
        dlogit = ((p - y) / B).reshape(-1, 1)
    return loss, dlogit, logit, cache, p

## Sanity Check

In [10]:
init_params(**VARS["ReGLU, param-matched"])
rs = np.random.RandomState(1); Xc, yc = rs.randn(2, IN_DIM), np.array([0.0, 1.0]); PW = 3.0
G0 = {k: np.zeros_like(v) for k, v in P_.items()}
_, da, _, cache_a, _ = fwd_loss(Xc, yc, False, True); model_bwd(da, cache_a, G0)
for key, idx in [("tokW", (0, 0)), (f"F1{LAYERS-1}", (3, 5)), (f"F1{LAYERS-1}", (3, HID + 5)), ("F20", (7, 2)), ("Wq0", (1, 1))]:
    e = 1e-5; old = P_[key][idx]
    P_[key][idx] = old + e; lp = fwd_loss(Xc, yc, False, True)[0]
    P_[key][idx] = old - e; lm = fwd_loss(Xc, yc, False, True)[0]
    P_[key][idx] = old
    num, ana = (lp - lm) / (2 * e), G0[key][idx]
    err = abs(num - ana) / (abs(num) + abs(ana) + 1e-12)
    print(f"check {key}{list(idx)}: numeric {num:.6e} analytic {ana:.6e} | relerr {err:.2e}")
    assert err < 1e-4, "backward pass does not match the numeric slope"

check tokW[0, 0]: numeric -6.215682e-02 analytic -6.215682e-02 | relerr 7.71e-11
check F11[3, 5]: numeric 0.000000e+00 analytic 0.000000e+00 | relerr 0.00e+00
check F11[3, 48]: numeric 0.000000e+00 analytic 0.000000e+00 | relerr 0.00e+00
check F20[7, 2]: numeric -6.228742e-02 analytic -6.228742e-02 | relerr 2.84e-10
check Wq0[1, 1]: numeric -8.219848e-03 analytic -8.219848e-03 | relerr 2.93e-09


## Training

In [11]:
import pickle, time
LR0, WD, B1, B2, EP = 1e-3, 1e-4, 0.9, 0.999, 1e-8
def eval_all(X, y):
    return np.concatenate([fwd_loss(X[i:i+256], y[i:i+256], False, False)[4] for i in range(0, len(X), 256)])

def train_scratch(Xa, ya, spec, E, seed):
    """Adam + cosine for E epochs (E chosen by the pytorch twin's inner validation on this fold). Leaves the trained weights in P_."""
    np.random.seed(seed); init_params(**spec)
    M = {k: np.zeros_like(a) for k, a in P_.items()}; V = {k: np.zeros_like(a) for k, a in P_.items()}
    N = len(Xa); step = 0
    for epoch in range(1, E + 1):
        lr = 0.5 * LR0 * (1 + np.cos(np.pi * (epoch - 1) / 100))
        perm = np.random.permutation(N)
        for i in range(0, N, 64):
            idx = perm[i:i+64]
            loss, dlogit, _, cache, _ = fwd_loss(Xa[idx], ya[idx], True, True)
            G = {k: np.zeros_like(a) for k, a in P_.items()}
            model_bwd(dlogit, cache, G); step += 1
            for k in P_:
                g = G[k] + WD * P_[k]
                M[k] = B1 * M[k] + (1 - B1) * g; V[k] = B2 * V[k] + (1 - B2) * (g ** 2)
                mh = M[k] / (1 - B1 ** step); vh = V[k] / (1 - B2 ** step)
                P_[k] -= lr * (mh / (np.sqrt(vh) + EP))

last = {}                                    # trained weights of the last fold this job ran, for the Parity Check
for fold, (a, b) in enumerate(StratifiedKFold(K, shuffle=True, random_state=SEED).split(X_all, y_all), start=1):
    if fold not in FOLDS:
        continue
    cache = f"weights_V4_1/_progress_classification_scratch_f{fold}.pkl"
    if os.path.exists(cache):
        continue
    Xa, Xb, ya, yb = prep_split(a, b); Xa, Xb, ya, yb = Xa.astype(np.float64), Xb.astype(np.float64), ya.astype(np.float64), yb.astype(np.float64)
    PW = (ya == 0).sum() / (ya == 1).sum()
    rows, sc = [], {}
    for vname, spec in VARS.items():
        t0 = time.time(); o_ = ORA[(NAME + vname, 1, fold)]
        train_scratch(Xa, ya, spec, int(o_["epochs"]), SEED + 7 + fold)
        s = eval_all(Xb, yb); thr = float(o_["threshold"]); pred = (s >= thr).astype(int)
        rows.append(dict(model=NAME + vname, tier="extracurricular", selection="inner validation", repeat=1, fold=fold, n_test=int(len(yb)),
            f1=round(float(f1_score(yb, pred, zero_division=0)), 4), accuracy=round(float(accuracy_score(yb, pred)), 4),
            precision=round(float(precision_score(yb, pred, zero_division=0)), 4), recall=round(float(recall_score(yb, pred, zero_division=0)), 4),
            roc_auc=round(float(roc_auc_score(yb, s)), 4), bce=round(float(log_loss(yb, np.clip(s, 1e-12, 1 - 1e-12))), 4),
            threshold=thr, epochs=int(o_["epochs"]), n_params=int(sum(x.size for x in P_.values())), seconds=round(time.time() - t0, 1)))
        sc[f"{NAME + vname}|r1"] = (b, s)
        last[vname] = dict(spec=spec, P={k: x.copy() for k, x in P_.items()}, Xb=Xb, yb=yb, PW=PW)
        print(f"fold {fold} {vname:22s} F1 {rows[-1]['f1']:.4f} (pytorch {o_['f1']:.4f}) | AUC {rows[-1]['roc_auc']:.4f} (pytorch {o_['roc_auc']:.4f}) | {rows[-1]['seconds']:.0f}s", flush=True)
    pickle.dump({"rows": rows, "scores": sc}, open(cache, "wb"))

fold 5 ReLU (baseline)        F1 0.5128 (pytorch 0.5588) | AUC 0.8667 (pytorch 0.8681) | 19s


fold 5 ReGLU, param-matched   F1 0.5432 (pytorch 0.5031) | AUC 0.8743 (pytorch 0.8405) | 50s


## K-Fold Summary

In [12]:
import glob
parts = [pickle.load(open(f, "rb")) for f in sorted(glob.glob("weights_V4_1/_progress_classification_scratch_f*.pkl"))]
rows = [r for p in parts for r in p["rows"]]
if len(parts) < K:
    print(f"{len(parts)}/{K} folds finished so far; the job that finishes the last fold writes the summary")
else:
    kdf = pd.DataFrame(rows)
    KEYS = ['f1', 'accuracy', 'precision', 'recall', 'roc_auc', 'bce']
    summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g} for m, g in kdf.groupby("model")}
    print(kdf.groupby("model")["f1"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4).to_string())
    scores = {}
    for p in parts:
        for key, (idx, s) in p["scores"].items():
            scores.setdefault(key, np.full(len(y_all), np.nan))[idx] = s
    json.dump({"k": K, "repeats": REPEATS, "folds": rows, "summary": summary}, open("weights_V4_1/classification_scratch_kfold_metrics.json", "w"), indent=2, default=str)
    np.savez("weights_V4_1/classification_scratch_kfold_scores.npz", **scores)
    pt = pd.DataFrame([r for r in ORA.values() if r["repeat"] == 1 and r["model"] in {NAME + v for v in VARS}])
    print("pytorch twin, same repeat (mean F1):", pt.groupby("model")["f1"].mean().round(4).to_dict())

                                              mean     std
model                                                     
FT-Transformer-lite | ReGLU, param-matched  0.5423  0.0458
FT-Transformer-lite | ReLU (baseline)       0.5356  0.0156
pytorch twin, same repeat (mean F1): {'FT-Transformer-lite | ReGLU, param-matched': 0.5003, 'FT-Transformer-lite | ReLU (baseline)': 0.5456}


## Parity Check

In [13]:
import warnings
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             log_loss, average_precision_score, mean_squared_error, mean_absolute_error, r2_score)
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

device = torch.device("cpu")


class TransformerBlock(nn.Module):
    """Pre-norm block. ffn='relu': Linear(d,hid) -> ReLU -> Linear(hid,d).
       ffn='reglu': Linear(d,2*hid) split into (value, gate) -> value * ReLU(gate) -> Linear(hid,d)."""
    def __init__(self, d=32, h=4, p=0.1, ffn="relu", hid=None):
        super().__init__()
        assert d % h == 0; self.h, self.dk, self.ffn = h, d // h, ffn
        hid = hid or 2 * d
        self.ln1 = nn.LayerNorm(d, eps=1e-5); self.ln2 = nn.LayerNorm(d, eps=1e-5)
        self.wq = nn.Linear(d, d); self.wk = nn.Linear(d, d); self.wv = nn.Linear(d, d); self.wo = nn.Linear(d, d)
        self.fc1 = nn.Linear(d, hid if ffn == "relu" else 2 * hid); self.fc2 = nn.Linear(hid, d)
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p); self.attn_drop = nn.Dropout(p)
        for fc in (self.wq, self.wk, self.wv, self.wo, self.fc1, self.fc2):
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu"); nn.init.zeros_(fc.bias)
    def forward(self, t):
        B, T, D = t.shape
        h_in = self.ln1(t)
        Q = self.wq(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        K = self.wk(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        V = self.wv(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        s = (Q @ K.transpose(-2, -1)) / (self.dk ** 0.5)
        p = self.attn_drop(torch.softmax(s, dim=-1))
        o = (p @ V).transpose(1, 2).reshape(B, T, D)
        u = t + self.wo(o)
        z = self.fc1(self.ln2(u))
        if self.ffn == "reglu":
            val, gate = z.chunk(2, dim=-1); a = val * self.relu(gate)
        else:
            a = self.relu(z)
        return u + self.fc2(self.drop(a))

class FTTransformerLite(nn.Module):
    def __init__(self, in_dim=22, d=32, layers=2, heads=4, p=0.1, ffn="relu", hid=None):
        super().__init__()
        # tokenizer: feature j has its own Linear(1 -> d). Stored as two tables (in_dim x d) so all
        # features are embedded in one operation; the maths is identical to in_dim separate Linear(1, d).
        self.tok_w = nn.Parameter(torch.randn(in_dim, d) * (2.0 ** 0.5))   # He init, fan_in = 1
        self.tok_b = nn.Parameter(torch.zeros(in_dim, d))
        self.cls = nn.Parameter(torch.randn(d) * 0.02)
        self.blocks = nn.ModuleList([TransformerBlock(d, heads, p, ffn, hid) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(d, eps=1e-5)
        self.head = nn.Linear(d, 1)
        nn.init.kaiming_normal_(self.head.weight, nonlinearity="linear"); nn.init.zeros_(self.head.bias)
    def forward(self, x):
        B = x.shape[0]
        t = x.unsqueeze(-1) * self.tok_w + self.tok_b                       # (B, in_dim, d)
        t = torch.cat([self.cls.view(1, 1, -1).expand(B, 1, -1), t], dim=1)
        for blk in self.blocks:
            t = blk(t)
        return self.head(self.ln_f(t[:, 0]))

# hidden size 64 = 2d is the ReLU baseline. ReGLU "same hidden" keeps 64 (more parameters);
# ReGLU "param-matched" uses hidden = 4/3 d = 43 (the original paper's factor) to land on the ReLU parameter count.
VARIANTS = {
    "ReLU (baseline)":       dict(ffn="relu", hid=64),
    "ReGLU, same hidden":    dict(ffn="reglu", hid=64),
    "ReGLU, param-matched":  dict(ffn="reglu", hid=43),
}
def make_model(v, seed, in_dim=None):
    torch.manual_seed(seed)
    return FTTransformerLite(in_dim=in_dim or len(FEATURE_COLS), **VARIANTS[v]).to(device)


def load_into_torch(m, Pn):
    """Copy the scratch arrays into the PyTorch twin (Linear weights are stored transposed)."""
    with torch.no_grad():
        m.tok_w.copy_(torch.tensor(Pn["tokW"])); m.tok_b.copy_(torch.tensor(Pn["tokb"])); m.cls.copy_(torch.tensor(Pn["cls"]))
        for l, b in enumerate(m.blocks):
            b.ln1.weight.copy_(torch.tensor(Pn[f"g1{l}"])); b.ln1.bias.copy_(torch.tensor(Pn[f"b1{l}"]))
            b.ln2.weight.copy_(torch.tensor(Pn[f"g2{l}"])); b.ln2.bias.copy_(torch.tensor(Pn[f"b2{l}"]))
            for nm, lin in (("q", b.wq), ("k", b.wk), ("v", b.wv), ("o", b.wo)):
                lin.weight.copy_(torch.tensor(Pn[f"W{nm}{l}"].T)); lin.bias.copy_(torch.tensor(Pn[f"b{nm}{l}"]))
            b.fc1.weight.copy_(torch.tensor(Pn[f"F1{l}"].T)); b.fc1.bias.copy_(torch.tensor(Pn[f"f1b{l}"]))
            b.fc2.weight.copy_(torch.tensor(Pn[f"F2{l}"].T)); b.fc2.bias.copy_(torch.tensor(Pn[f"f2b{l}"]))
        m.ln_f.weight.copy_(torch.tensor(Pn["gf"])); m.ln_f.bias.copy_(torch.tensor(Pn["bf"]))
        m.head.weight.copy_(torch.tensor(Pn["Wh"].T)); m.head.bias.copy_(torch.tensor(Pn["bh"]))

name_map = {"tok_w": "tokW", "tok_b": "tokb", "cls": "cls", "ln_f.weight": "gf", "ln_f.bias": "bf", "head.weight": "Wh", "head.bias": "bh"}
for l in range(LAYERS):
    name_map.update({f"blocks.{l}.ln1.weight": f"g1{l}", f"blocks.{l}.ln1.bias": f"b1{l}", f"blocks.{l}.ln2.weight": f"g2{l}", f"blocks.{l}.ln2.bias": f"b2{l}"})
    for nm in "qkvo":
        name_map.update({f"blocks.{l}.w{nm}.weight": f"W{nm}{l}", f"blocks.{l}.w{nm}.bias": f"b{nm}{l}"})
    name_map.update({f"blocks.{l}.fc1.weight": f"F1{l}", f"blocks.{l}.fc1.bias": f"f1b{l}", f"blocks.{l}.fc2.weight": f"F2{l}", f"blocks.{l}.fc2.bias": f"f2b{l}"})

for vname, r_ in last.items():
    spec, P_, xb, yb_, PW = r_["spec"], r_["P"], r_["Xb"][:64], r_["yb"][:64], r_["PW"]
    FFN, HID = spec["ffn"], spec["hid"]; FFW = HID if FFN == "relu" else 2 * HID
    m = make_model(vname, 0, IN_DIM).double().eval(); load_into_torch(m, P_)
    loss_t = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([PW], dtype=torch.float64))(m(torch.tensor(xb)), torch.tensor(yb_).view(-1, 1)); loss_t.backward()
    loss_s, dlogit, logit_s, cache, _ = fwd_loss(xb, yb_, False, True)
    G = {k: np.zeros_like(a) for k, a in P_.items()}; model_bwd(dlogit, cache, G)
    d_logit = float(np.abs(m(torch.tensor(xb)).detach().numpy() - logit_s).max())
    d_loss = abs(float(loss_t) - float(loss_s))
    d_grad, gmax = 0.0, max(float(p.grad.abs().max()) for p in m.parameters())
    for pn, p in m.named_parameters():
        g_s = G[name_map[pn]]; g_t = p.grad.numpy()
        g_s = g_s.T if p.dim() == 2 and pn not in ("tok_w", "tok_b") else g_s
        d_grad = max(d_grad, float(np.abs(g_t - g_s).max() / gmax))
    print(f"{vname:22s} | max|dlogit| {d_logit:.2e} | dloss {d_loss:.2e} | max rel grad diff {d_grad:.2e}")
    assert d_logit < 1e-8 and d_loss < 1e-8 and d_grad < 1e-6, "scratch and PyTorch disagree on identical weights"
print("scratch forward and backward match PyTorch on identical weights (last fold this job trained)")


ReLU (baseline)        | max|dlogit| 5.44e-15 | dloss 4.44e-16 | max rel grad diff 2.32e-15
ReGLU, param-matched   | max|dlogit| 2.26e-14 | dloss 5.55e-16 | max rel grad diff 3.96e-15
scratch forward and backward match PyTorch on identical weights (last fold this job trained)
